
 GOLD LAYER — FACT SALES

 This notebook builds the core business table: gold.fact_sales

 Grain:
   1 row = 1 order line (product-level transaction)

 Sources:
   - silver.sales_orders
   - silver.crm_customers
   - silver.products
   - silver.stores

 Business usage:
  - Revenue analysis
  - Customer behavior
  - Product performance


1. Load Silver Tables

In [0]:

df_sales = spark.table("silver.sales_orders")
df_products = spark.table("silver.products")
df_customers = spark.table("silver.crm_customers")

2. Basic Validation

In [0]:


print("Sales count:", df_sales.count())
print("Products count:", df_products.count())
print("Customers count:", df_customers.count())

3. Build Fact Table (Joins)

In [0]:


df_joined = df_sales \
    .join(df_products, "product_id", "left") \
    .join(df_customers, "customer_id", "left")

4. Select Business Columns

In [0]:


from pyspark.sql.functions import col

df_fact = df_joined.select(
    # Core identifiers
    col("order_id"),
    col("order_date"),
    col("customer_id"),
    col("product_id"),

    # Measures
    col("quantity"),
    col("unit_price"),
    col("total_amount"),

    # Status & time
    col("status"),
    col("order_year"),
    col("order_month"),

    # Customer attributes
    col("customer_name"),
    col("loyalty_tier"),

    # Product attributes
    col("product_name"),
    col("category")
)

5. Derived Columns

In [0]:

df_fact = df_fact.withColumn("revenue", col("total_amount"))

6. Audit Columns

In [0]:

from pyspark.sql.functions import current_timestamp

df_fact = df_fact.withColumn("created_at", current_timestamp())

7. Data Quality Checks

In [0]:

from pyspark.sql.functions import col

null_customers = df_fact.filter(col("customer_id").isNull()).count()
null_products = df_fact.filter(col("product_id").isNull()).count()

print(f"Null customer_id: {null_customers}")
print(f"Null product_id: {null_products}")

8. Write to Gold Table

In [0]:

spark.sql("DROP TABLE IF EXISTS gold.fact_sales")

df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_sales")

9. Validation

In [0]:

display(spark.table("gold.fact_sales"))